In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# ============================================================
# TVP-VAR GFEVD -> Connectedness
# ============================================================
# Assumption
# - GFEVD matrix theta[t, i, j]
# - row i = response variable
# - col j = shock source
#
# Outputs
# - connectedness_time.csv
# - connectedness_mean.csv
# - connectedness_table_mean.csv
# - net_direction.csv
# - pairwise_net_mean.csv
# - tci_series.png
# - to_series.png
# - from_series.png
# - net_series.png
# ============================================================

BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "connectedness_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

DEFAULT_VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "dlog_OIL",
    "dlog_GAS",
    "d_UST10Y",
    "d_VIX",
]

EPS = 1e-12


# ============================================================
# Helper functions
# ============================================================

def infer_gfevd_tnn(arr: np.ndarray) -> np.ndarray:
    """
    Return GFEVD as shape (T, N, N).
    Accepts either (T, N, N) or (N, N, T).
    """
    if arr.ndim != 3:
        raise ValueError(f"gfevd_all must be 3D, got shape={arr.shape}")

    # Case 1: already (T, N, N)
    if arr.shape[1] == arr.shape[2] and arr.shape[0] != arr.shape[1]:
        return arr.astype(float, copy=True)

    # Case 2: (N, N, T)
    if arr.shape[0] == arr.shape[1] and arr.shape[2] != arr.shape[0]:
        return np.transpose(arr, (2, 0, 1)).astype(float, copy=True)

    # Ambiguous but square across all dims
    if arr.shape[1] == arr.shape[2]:
        return arr.astype(float, copy=True)

    raise ValueError(f"Cannot infer GFEVD shape. Expected (T,N,N) or (N,N,T), got {arr.shape}")


def infer_var_names(n: int) -> list:
    """
    Use DEFAULT_VAR_NAMES if length matches.
    Otherwise generate generic names.
    """
    if len(DEFAULT_VAR_NAMES) == n:
        return DEFAULT_VAR_NAMES

    print(f"[WARN] DEFAULT_VAR_NAMES length={len(DEFAULT_VAR_NAMES)} does not match N={n}.")
    print("[WARN] Using generic variable names.")
    return [f"VAR{i+1}" for i in range(n)]


def load_dates(target_len: int):
    """
    Read Date column from candidate files and align by taking last target_len rows.
    """
    for fp in DATE_FILE_CANDIDATES:
        if not fp.exists():
            continue

        try:
            df = pd.read_csv(fp)

            date_col = None
            for c in ["Date", "date", "DATE"]:
                if c in df.columns:
                    date_col = c
                    break

            if date_col is None:
                continue

            dt = pd.to_datetime(df[date_col], errors="coerce")
            dt = dt.dropna().reset_index(drop=True)

            if len(dt) < target_len:
                print(f"[WARN] {fp} has only {len(dt)} valid dates, target_len={target_len}. Skipped.")
                continue

            return dt.iloc[-target_len:].reset_index(drop=True)

        except Exception as e:
            print(f"[WARN] Failed to read dates from {fp}: {e}")

    return None


def normalize_gfevd_to_percent(gfevd_tnn: np.ndarray) -> np.ndarray:
    """
    Normalize each row to sum to 100.
    Handles both raw proportion-scale GFEVD and percent-scale GFEVD.
    """
    arr = np.asarray(gfevd_tnn, dtype=float).copy()

    if not np.all(np.isfinite(arr)):
        raise ValueError("GFEVD contains NaN or inf values.")

    # Negative values should not exist in GFEVD.
    min_val = arr.min()
    if min_val < -1e-10:
        raise ValueError(f"GFEVD contains negative values. min={min_val}")

    arr[arr < 0] = 0.0

    row_sum = arr.sum(axis=2, keepdims=True)

    if np.any(row_sum <= EPS):
        bad = np.argwhere(row_sum.squeeze(-1) <= EPS)
        raise ValueError(f"Some GFEVD rows have zero row sum. First bad index={bad[0].tolist()}")

    theta_pct = arr / row_sum * 100.0

    max_row_err = np.max(np.abs(theta_pct.sum(axis=2) - 100.0))
    if max_row_err > 1e-8:
        raise RuntimeError(f"Row normalization failed. max_row_err={max_row_err}")

    return theta_pct


def compute_connectedness(theta_pct: np.ndarray):
    """
    theta_pct: (N, N), row=response, col=shock, each row sums to 100.

    FROM_i = sum_{j != i} theta[i, j]
    TO_i   = sum_{j != i} theta[j, i]
    NET_i  = TO_i - FROM_i
    TCI    = sum off-diagonal / N
    """
    if theta_pct.ndim != 2 or theta_pct.shape[0] != theta_pct.shape[1]:
        raise ValueError(f"theta_pct must be square 2D matrix, got {theta_pct.shape}")

    n = theta_pct.shape[0]
    diag = np.diag(theta_pct)

    from_ = theta_pct.sum(axis=1) - diag
    to_ = theta_pct.sum(axis=0) - diag
    net_ = to_ - from_
    tci = (theta_pct.sum() - np.trace(theta_pct)) / n

    return from_, to_, net_, tci


def compute_pairwise_net(theta_pct: np.ndarray):
    """
    pairwise_net[i, j] = i -> j minus j -> i
                       = theta[j, i] - theta[i, j]
    Positive value means row variable i is net transmitter to column variable j.
    """
    return theta_pct.T - theta_pct


def save_series_plot(df, cols, title, ylabel, out_path, x_col):
    plt.figure(figsize=(14, 5))

    x = df[x_col]
    if x_col == "Date":
        x = pd.to_datetime(x)

    for c in cols:
        label = c.split("_", 1)[1] if "_" in c else c
        plt.plot(x, df[c], linewidth=1.1, label=label)

    if "NET" in title:
        plt.axhline(0.0, linestyle="--", linewidth=1.0)

    plt.title(title)
    plt.xlabel(x_col)
    plt.ylabel(ylabel)
    plt.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


# ============================================================
# Load and normalize GFEVD
# ============================================================

if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE.resolve()}")

raw_gfevd = np.load(GFEVD_FILE)
gfevd_tnn = infer_gfevd_tnn(raw_gfevd)

T_eff, N, N2 = gfevd_tnn.shape
if N != N2:
    raise ValueError(f"GFEVD last two dimensions must be square, got {gfevd_tnn.shape}")

VAR_NAMES = infer_var_names(N)

print(f"[INFO] Loaded GFEVD shape: {raw_gfevd.shape}")
print(f"[INFO] Using GFEVD as shape: {gfevd_tnn.shape}")
print(f"[INFO] Variables: {VAR_NAMES}")

theta_pct_all = normalize_gfevd_to_percent(gfevd_tnn)

print("[INFO] Row normalization complete.")
print(f"[INFO] Max row-sum error: {np.max(np.abs(theta_pct_all.sum(axis=2) - 100.0)):.12f}")


# ============================================================
# Dates
# ============================================================

dates = load_dates(T_eff)

if dates is not None:
    index_col = "Date"
    index_values = dates
else:
    index_col = "t"
    index_values = np.arange(T_eff)

print(f"[INFO] Index column: {index_col}")


# ============================================================
# Time-varying connectedness
# ============================================================

records = []
pairwise_net_all = np.zeros((T_eff, N, N), dtype=float)

for t in range(T_eff):
    theta = theta_pct_all[t]

    from_, to_, net_, tci = compute_connectedness(theta)
    pairwise_net_all[t] = compute_pairwise_net(theta)

    row = {index_col: index_values.iloc[t] if index_col == "Date" else int(index_values[t])}

    for i, name in enumerate(VAR_NAMES):
        row[f"FROM_{name}"] = from_[i]
        row[f"TO_{name}"] = to_[i]
        row[f"NET_{name}"] = net_[i]

    row["TCI"] = tci
    records.append(row)

df_time = pd.DataFrame(records)


# ============================================================
# Mean summary
# ============================================================

rows_mean = []

for name in VAR_NAMES:
    rows_mean.append({
        "Variable": name,
        "FROM_mean": df_time[f"FROM_{name}"].mean(),
        "TO_mean": df_time[f"TO_{name}"].mean(),
        "NET_mean": df_time[f"NET_{name}"].mean(),
    })

df_mean = pd.DataFrame(rows_mean)

system_row = pd.DataFrame([{
    "Variable": "SYSTEM",
    "FROM_mean": np.nan,
    "TO_mean": np.nan,
    "NET_mean": np.nan,
    "TCI_mean": df_time["TCI"].mean(),
}])

df_mean = pd.concat([df_mean, system_row], ignore_index=True)


# ============================================================
# Mean connectedness table like DY table
# Rows: response
# Cols: shock source
# FROM: row off-diagonal sum
# TO row: column off-diagonal sum
# NET row: TO - FROM
# ============================================================

theta_mean = theta_pct_all.mean(axis=0)

from_mean, to_mean, net_mean, tci_mean = compute_connectedness(theta_mean)

df_table = pd.DataFrame(theta_mean, index=VAR_NAMES, columns=VAR_NAMES)
df_table["FROM"] = from_mean

to_row = pd.Series(list(to_mean) + [np.nan], index=list(VAR_NAMES) + ["FROM"], name="TO")
net_row = pd.Series(list(net_mean) + [tci_mean], index=list(VAR_NAMES) + ["FROM"], name="NET/TCI")

df_connectedness_table = pd.concat([df_table, to_row.to_frame().T, net_row.to_frame().T])


# ============================================================
# Net direction ranking
# ============================================================

df_net = (
    df_mean[df_mean["Variable"] != "SYSTEM"][["Variable", "NET_mean"]]
    .sort_values("NET_mean", ascending=False)
    .reset_index(drop=True)
)


# ============================================================
# Pairwise net mean
# ============================================================

pairwise_net_mean = pairwise_net_all.mean(axis=0)
df_pairwise = pd.DataFrame(pairwise_net_mean, index=VAR_NAMES, columns=VAR_NAMES)


# ============================================================
# Save CSV
# ============================================================

out_time = OUT_DIR / "connectedness_time.csv"
out_mean = OUT_DIR / "connectedness_mean.csv"
out_table = OUT_DIR / "connectedness_table_mean.csv"
out_net = OUT_DIR / "net_direction.csv"
out_pairwise = OUT_DIR / "pairwise_net_mean.csv"

df_time.to_csv(out_time, index=False)
df_mean.to_csv(out_mean, index=False)
df_connectedness_table.to_csv(out_table, index=True)
df_net.to_csv(out_net, index=False)
df_pairwise.to_csv(out_pairwise, index=True)

print("[INFO] Saved CSV files:")
print(f"  - {out_time}")
print(f"  - {out_mean}")
print(f"  - {out_table}")
print(f"  - {out_net}")
print(f"  - {out_pairwise}")


# ============================================================
# Save plots
# ============================================================

x_col = index_col

out_tci_png = OUT_DIR / "tci_series.png"
out_from_png = OUT_DIR / "from_series.png"
out_to_png = OUT_DIR / "to_series.png"
out_net_png = OUT_DIR / "net_series.png"

plt.figure(figsize=(12, 4))
x = pd.to_datetime(df_time[x_col]) if x_col == "Date" else df_time[x_col]
plt.plot(x, df_time["TCI"], linewidth=1.5)
plt.title("Total Connectedness Index (TCI)")
plt.xlabel(x_col)
plt.ylabel("TCI (%)")
plt.tight_layout()
plt.savefig(out_tci_png, dpi=300, bbox_inches="tight")
plt.close()

from_cols = [f"FROM_{name}" for name in VAR_NAMES]
to_cols = [f"TO_{name}" for name in VAR_NAMES]
net_cols = [f"NET_{name}" for name in VAR_NAMES]

save_series_plot(df_time, from_cols, "Directional FROM (Incoming Spillover)", "Contribution (%)", out_from_png, x_col)
save_series_plot(df_time, to_cols, "Directional TO (Outgoing Spillover)", "Contribution (%)", out_to_png, x_col)
save_series_plot(df_time, net_cols, "NET Directional Connectedness", "NET (%)", out_net_png, x_col)

print("[INFO] Saved plot files:")
print(f"  - {out_tci_png}")
print(f"  - {out_from_png}")
print(f"  - {out_to_png}")
print(f"  - {out_net_png}")


# ============================================================
# Diagnostics
# ============================================================

print("\n[INFO] Average NET ranking")
print(df_net.to_string(index=False))

print("\n[INFO] Average TCI")
print(f"TCI_mean = {df_time['TCI'].mean():.4f}%")

print("\n[INFO] Check identities")
print(f"Mean sum(TO)   = {df_mean[df_mean['Variable'] != 'SYSTEM']['TO_mean'].sum():.6f}")
print(f"Mean sum(FROM) = {df_mean[df_mean['Variable'] != 'SYSTEM']['FROM_mean'].sum():.6f}")
print(f"N * TCI_mean   = {N * df_time['TCI'].mean():.6f}")

[INFO] Loaded GFEVD shape: (1031, 9, 9)
[INFO] Using GFEVD as shape: (1031, 9, 9)
[INFO] Variables: ['dlog_SOLVPN', 'dlog_COPPER', 'dlog_GOLD', 'dlog_SILVER', 'dlog_DXY', 'dlog_OIL', 'dlog_GAS', 'd_UST10Y', 'd_VIX']
[INFO] Row normalization complete.
[INFO] Max row-sum error: 0.000000000000
[INFO] Index column: Date
[INFO] Saved CSV files:
  - connectedness_output\connectedness_time.csv
  - connectedness_output\connectedness_mean.csv
  - connectedness_output\connectedness_table_mean.csv
  - connectedness_output\net_direction.csv
  - connectedness_output\pairwise_net_mean.csv
[INFO] Saved plot files:
  - connectedness_output\tci_series.png
  - connectedness_output\from_series.png
  - connectedness_output\to_series.png
  - connectedness_output\net_series.png

[INFO] Average NET ranking
   Variable   NET_mean
dlog_SOLVPN   9.851569
dlog_SILVER   6.935901
      d_VIX   5.142845
  dlog_GOLD   4.882330
   dlog_DXY   0.386122
dlog_COPPER  -2.392434
   d_UST10Y  -4.752491
   dlog_GAS  -9.82102